# Proposta 1 — F1-macro 500 m dos MESMOS 600 pontos cegos

Versão corrigida: abre diretamente os arquivos do Drive montado, sem copiar nem apagar originais. Executa uma região por vez, exibe os erros completos e salva logs e resultados a cada etapa. As referências externas são pseudo-rótulos CHMv2, não verdade-terreno humana.

In [ ]:
!apt-get update -qq && apt-get install -y -qq p7zip-full >/dev/null
!pip -q install numpy pandas rasterio scikit-image scikit-learn requests gdown

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
originais=Path('/content/drive/MyDrive/Imagens do Mestrado')
files=['SAGRADA_FAMÍLIA_CAXIAS DO SUL.zip','LOMBA_GRANDE_NOVO_HAMBURGO.zip','CRISTO_REI_SÃO_LEOPOLDO.rar']
for name in files:
    p=originais/name
    print('OK' if p.is_file() else 'FALTA',p)
    assert p.is_file(), f'Ajuste o caminho para o arquivo original no seu Drive: {p}'

In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path
repo=Path('/content/mestrado-arvores-urbanas')
if not (repo/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--branch','feature/validacao-geografica-externa',
        'https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch','origin','feature/validacao-geografica-externa'],check=True)
    subprocess.run(['git','-C',str(repo),'checkout','feature/validacao-geografica-externa'],check=True)
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
script=repo/'experimentos/validacao_externa_regioes/chmv2_verificacao/avaliar_600_pontos_C_vs_A_500m.py'
assert script.is_file(),script
destino=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/resultado_final_600_pontos_500m')
destino.mkdir(parents=True,exist_ok=True)
regioes=['sagrada_familia_caxias','lomba_grande_novo_hamburgo','cristo_rei_sao_leopoldo']
situacao={}
for regiao in regioes:
    out=Path('/content')/('f1_600_500m_'+regiao)
    env=os.environ.copy()
    env['CHM_ORIGINAIS_DRIVE']=str(originais)
    env['CHM_F1_OUTPUT']=str(out)
    env['CHM_REGION_ONLY']=regiao
    print('REGIÃO:',regiao,'| Arquivos originais lidos DIRETAMENTE do Drive montado.',flush=True)
    process=subprocess.Popen([sys.executable,'-u',str(script)],cwd=str(repo),env=env,
            stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    linhas=[]
    for linha in process.stdout:
        print(linha,end='',flush=True)
        linhas.append(linha)
    exitcode=process.wait()
    logfile=destino/(regiao+'_execucao_completa.log')
    logfile.write_text(''.join(linhas),encoding='utf-8')
    if out.is_dir():
        for f in out.glob('*.csv'):
            shutil.copy2(f,destino/(regiao+'_'+f.name))
        for f in out.glob('*.json'):
            shutil.copy2(f,destino/(regiao+'_'+f.name))
    situacao[regiao]=(exitcode,str(out))
    print('RESULTADO REGIÃO:',regiao,'OK' if exitcode==0 else 'ERRO: '+str(exitcode),'| LOG:',logfile,flush=True)
print('EXECUÇÕES:',situacao)


In [ ]:
import json, pandas as pd
partes=[]
for regiao,(status,pasta) in situacao.items():
    caminho=Path(pasta)/'COMPARACAO_F1_MACRO_500m_600_PONTOS.csv'
    if caminho.is_file():
        partes.append(pd.read_csv(caminho))
if not partes:
    print('BLOQUEIO: Nenhuma região calculou F1. Verifique os arquivos *_execucao_completa.log no Drive.')
else:
    todos=pd.concat(partes,ignore_index=True)
    resumo=todos.pivot(index='regiao',columns='modelo',values='f1_macro')
    resumo['diferenca_C_menos_A']=resumo['C_retreinado_371']-resumo['A_raster_original']
    print(resumo.to_string(float_format=lambda v:f'{v:.4f}'))
    completo=destino/'COMPARACAO_FINAL_F1_500m_MESMOS_600_PONTOS.csv'
    resumo.to_csv(completo)
    print('RESULTADOS CONSOLIDADOS SALVOS:',completo)
    for regiao,(status,pasta) in situacao.items():
        file=Path(pasta)/'manifesto_comparacao.json'
        if file.is_file():
            m=json.loads(file.read_text(encoding='utf-8'))
            print('CONFERÊNCIA DO MODELO A:',regiao,m['status'],'divergências',m['diferencas_baseline_A'])
            if m['diferencas_baseline_A']:
                print('ATENÇÃO: desempenho C não validado contra a reprodução idêntica do A.')


In [ ]:
print('Pasta de evidências e resultados no Google Drive:',destino)
